# Sub-agent delegation

- Agent context can grow quickly as conversations progress, leading to several long context-related problems. 
- A primary issue is context clash or confusion, where mixed objectives within the same context window can lead to suboptimal performance. 
- Context isolation provides an effective solution by delegating tasks to specialized sub-agents, each operating within their own isolated context window. 
- This approach prevents context clashes, confusion, poisoning, and dilution while enabling focused, specialized task execution.

In [ ]:
from typing import Annotated, Any

from langchain.agents import AgentState, create_agent
from langchain.messages import ToolMessage
from langchain.tools import InjectedState, InjectedToolCallId, tool
from langgraph.types import Command
from loguru import logger
from tavily import TavilyClient

from chain_reaction.config import APIKeys, get_chat_model
from chain_reaction.reducers import reduce_dict
from chain_reaction.utils import format_messages

## Deep Agent State

- Add a list of files for agent to track items in filesystem

In [ ]:
class DeepAgentState(AgentState):
    """Extended agent state that includes a mapping of {filepath, content}."""

    files: Annotated[dict[str, str], reduce_dict]

## Filesystem tools

In [ ]:
@tool
def ls(state: Annotated[DeepAgentState, InjectedState]) -> list[str]:
    """List all files (filepaths) in agent accessible filesystem.

    Shows what files currently exist in agent memory. Use this to orient yourself
    before other file operations and maintain awareness of your file organization.

    Args:
        state (Annotated[DeepAgentState, InjectedState]): Injected agent state.

    Returns:
        list[str]: List of all files in filesystem, each item is a filepath.
    """
    files = list(state.get("files", {}).keys())
    logger.info("listing {num_files} file", num_files=len(files))
    return files


@tool
def read_file(
    filepath: str,
    state: Annotated[DeepAgentState, InjectedState],
    *,
    offset: int = 0,
    limit: int = 2_000,
    max_line_length: int = 2_000,
) -> str:
    """Read content from a file in the agent accessible filesystem.

    This tool returns file content with line numbers (like `cat -n`) and supports
    reading large files in chunks to avoid context overflow.

    ALWAYS read a file before editing it.

    Args:
        filepath (str): Filepath of file to read.
        state (Annotated[DeepAgentState, InjectedState]): Injected agent state.
        offset (int): Line number to start reading from. Defaults to 0.
        limit (int): Maximum number of lines to read. Defaults to 2000.
        max_line_length (int): Maximum line length to truncate any line at. Defaults to 2000.

    Returns:
        str: Formatted file content with line numbers, or error message if file not found
    """
    logger.info("Reading {filepath}[{offset}:+{limit}]", filepath=filepath, offset=offset, limit=limit)

    files: dict[str, str] = state.get("files", {})
    if filepath not in files:
        return f"Error: {filepath} not in filesystem. Available files: {list(files.keys())}"

    content = files[filepath]
    lines = content.splitlines()
    start_idx = offset
    end_idx = min(start_idx + limit, len(lines))

    if start_idx >= len(lines):
        return f"Error: Line offset {offset} exceeds file length ({len(lines)} lines)"

    result_lines = []
    for i in range(start_idx, end_idx):
        line_content = lines[i][:max_line_length]  # Truncate long lines
        result_lines.append(f"{i + 1:6d}\t{line_content}")

    return "\n".join(result_lines)


@tool
def write_file(
    filepath: str,
    content: str,
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """Generate command to write file to agent accessible filesystem.

    This tool creates new files or replaces entire file contents.
    Use for initial file creation or COMPLETE rewrites.
    IMPORTANT: This replaces the entire file content.

    Args:
        filepath (str): Path where the file should be created/updated
        content (str): Content to write to the file
        tool_call_id (Annotated[str, InjectedToolCallId]):
            Tool call identifier for message response (injected in tool node)

    Returns:
        Command: Command to update agent state with new file content
    """
    logger.info("Writing content to {filepath}", filepath=filepath)

    return Command(
        update={
            "files": {filepath: content},  # File will be combined with files in state
            "messages": [ToolMessage(f"Updated file {filepath}", tool_call_id=tool_call_id)],
        }
    )

## Web search subagent

In [ ]:
tavily_client = TavilyClient(api_key=APIKeys().tavily.get_secret_value())


@tool
def search_web(query: str) -> dict[str, Any]:
    """Performs a web search using Tavily.

    Args:
        query (str): The search query.

    Returns:
        dict[str, Any]: The search results.
    """
    logger.info("Running web query: {query}", query=query)
    return tavily_client.search(query=query)

In [ ]:
web_search_agent_system_prompt = f"""
You are research agent that uses web queries to find and save useful search results.

## Workflow Process
1. **Orient**: Use {ls.func.__name__} to see existing files before starting work
2. **Run search**: Use {search_web.func.__name__} to run a websearch for the user's query
3. **Save Results**: Use {write_file.func.__name__} to store the content of each search result. Every result should have its own file!
4. **List Results**: Provide the filename and a brief description of each result.
"""  # noqa: E501


web_search_agent = create_agent(
    model=get_chat_model(),
    tools=[search_web, ls, write_file],
    system_prompt=web_search_agent_system_prompt,
    state_schema=DeepAgentState,
)


@tool
def websearch_agent(
    query: str,
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """Invoke the web search subagent to conduct a websearch and save results to filesystem.

    Args:
        query (str): Query to search web for.
        state (Annotated[DeepAgentState, InjectedState]): Injected parent agent state.
        tool_call_id (Annotated[str, InjectedToolCallId]):
            Tool call identifier for message response (injected in tool node)
    """
    logger.info("Invoking websearch subagent for: {query}", query=query)

    # Create a narrowly scoped state for subagent to use
    sub_state = DeepAgentState(
        messages=[{"role": "user", "content": query}],
        files=state.get("files", {}),  # shared filesystem gets passed to subagent
    )

    # Invoke subagent with its narrow state and extract the last message from response
    response = web_search_agent.invoke(sub_state)
    messages = response.get("messages", [])
    content = messages[-1].content if messages else "Error: No response from websearch_agent"

    return Command(
        update={
            "files": response.get("files", {}),  # Merge any file changes
            "messages": [
                # Subagent response becomes tool message for parent
                ToolMessage(content, tool_call_id=tool_call_id)
            ],
        }
    )

## Main delegation agent

In [ ]:
main_system_prompt = f"""
You are a helpful question answering agent.

## Workflow Process
1. **Orient**: Use {ls.func.__name__} to see existing files before starting work
2. **Save Query**: Use {write_file.func.__name__} to store the user's query.
3. **Run Search**: Use {websearch_agent.func.__name__} to run searches and save results to filesystem. If you want to
run multiple searches, then dispatch multiple websearch_agent in parallel, with on query per agent.
4. **Read**: Once you are satisfied with the collected sources, use {read_file.func.__name__}
to retrieve the user's question AND any relevant saved files to ensure that you directly answer the user's question.

Only answer once you are 95% confident you have one or more results that provide a good answer.
It's your job to determine whether the user needs a detailed or short answer based on their query.
"""
agent = create_agent(
    model=get_chat_model(),
    tools=[websearch_agent, ls, write_file, read_file],
    system_prompt=main_system_prompt,
    state_schema=DeepAgentState,
)

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Give me a comprehensive summary of the Model Context Protocol (MCP) using FastMCP.",
        }
    ],
    "todos": [],
})

format_messages(result["messages"])

In [ ]:
result["files"].keys()